In [1]:
import sys
from pathlib import Path

# if notebook is in PRIN/notebooks, parent() is PRIN
project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))
print("Added to sys.path:", project_root)

import json
from utils.schema_json import ReportData, AnnotatedReport


Added to sys.path: C:\Users\lucatr\PycharmProjects\PRIN


In [2]:
# Parameters
TRAIN_FILE_NAME = "data_finetune_guido_openai_train.jsonl"
VALIDATION_FILE_NAME = "data_finetune_guido_openai_val.jsonl"
#FILE_NAMES = (TRAIN_FILE_NAME, VALIDATION_FILE_NAME)
TIPO = 'openai'

In [40]:
# Carichiamo i nostri file JSON
file_names = {
    'train': TRAIN_FILE_NAME,
    'validation': VALIDATION_FILE_NAME,
}

paths = {
    split: Path('../data/ft-dataset/' + file_name) for split, file_name in file_names.items()
}

data = dict()
for split, path in paths.items():
    with open(path, "r", encoding="utf-8") as f:
        data_list = [json.loads(line) for line in f]
        data[split] = data_list

train_data, validation_data = data['train'], data['validation']

print(f"{len(train_data) = }")
print(f"{type(train_data[0]) = }")
print(f"{type(train_data[0]['messages'][1]['content']) = }")  # Report text
print(f"{type(train_data[0]['messages'][2]['content']) = }")  # Annotations

len(train_data) = 116
type(train_data[0]) = <class 'dict'>
type(train_data[0]['messages'][1]['content']) = <class 'str'>
type(train_data[0]['messages'][2]['content']) = <class 'str'>


In [41]:
annotated_reports: dict[str, list[AnnotatedReport]] = {split: [] for split in file_names.keys()}
for split in annotated_reports:
    for record in data[split]:
        report_text = record['messages'][1]['content']
        if TIPO == 'openai':
            report_data = ReportData.model_validate_json(record['messages'][2]['content'])
        else:
            report_data = ReportData.model_validate(record['messages'][2]['content'])
        annotated_reports[split].append(AnnotatedReport(report_text=report_text, report_data=report_data))

Now that we have loaded our dataset, we will convert it to the proper desired format to upload for training.

The data will be converted to a jsonl format as follows:

*THIS IS AN EXAMPLE*
```json
{"text": "Avena e nocciole cioccolato fondente", "labels": {"food": ["sweet-snacks"], "country_label": "italy"}}
{"text": "Pomodori in pezzi", "labels": {"food": ["plant-based-foods-and-beverages"], "country_label": "belgium"}}
{"text": "Grandyoats, Nori Sesame Cashews", "labels": {"food": ["snacks"], "country_label": "united-states"}}
{"text": "Jus d'orange Profit", "labels": {"food": ["beverages", "plant-based-foods-and-beverages"], "country_label": "switzerland"}}
{"text": "Rote Beete", "labels": {"food": ["plant-based-foods", "plant-based-foods-and-beverages"], "country_label": "germany"}}
...
```
With an example of a label being:
```json
"labels": {
  # Chekbox like sedi_linfonodi
  "food": [
    "beverages",
    "plant-based-foods-and-beverages"
  ],
  # Drop-down like morfologia
  "country_label": "switzerland"
}
```
For multi-target classification.

In [56]:
def annotated_report_to_mistral_dict(annotated_report: AnnotatedReport):
    result = {
        'text': annotated_report.report_text,
        'labels': dict()
    }
    for k, v in annotated_report.report_data.model_dump().items():
        if not isinstance(v, dict):
            result['labels'][k] = v
        else:
            sub_list = []
            for kk, vv in v.items():
                if v[kk]:
                    sub_list.append(kk)
            result['labels'][k] = sub_list
    return result

In [60]:
mistral_data = dict()
for split, reports in annotated_reports.items():
    mistral_data[split] = [annotated_report_to_mistral_dict(report) for report in reports]

In [71]:
# Save the formatted data as JSONL files
for split, mistral_reports in mistral_data.items():
    with open(f'{split}_mistral_guido.jsonl', "w", encoding="utf-8") as f:
        for entry in mistral_reports:
            f.write(json.dumps(entry) + '\n')

print("JSONL files have been saved.")

JSONL files have been saved.
